## **Bronze Layer**

In [0]:
dbutils.fs.mount(
    source = "wasbs://retail@retailstoragesm.blob.core.windows.net",
    mount_point = "/mnt/retail",
    extra_configs = "fs.azure.account.key.manish040596.blob.core.windows.net": "MY_KEY"
)

In [0]:
dbutils.fs.ls('/mnt/retail_project/bronze/transaction/')

In [0]:
df_transactions = spark.read.parquet('/mnt/retail_project/bronze/transaction/')
df_products = spark.read.parquet('/mnt/retail_project/bronze/product/')
df_stores = spark.read.parquet('/mnt/retail_project/bronze/store/')

df_customers = spark.read.parquet('/mnt/retail_project/bronze/customer/manish040596/azure-data-engineer---multi-source/refs/heads/main/')
df_customers.display()

### **Silver Layer**

In [0]:
from pyspark.sql.functions import col

In [0]:
df_transactions = df_transactions.select(
    col("transaction_id").cast("int"),
    col("customer_id").cast("int"),
    col("product_id").cast("int"),
    col("store_id").cast("int"),
    col("quantity").cast("int"),
    col("transaction_date").cast("date")
)

df_products = df_products.select(
    col("product_id").cast("int"),
    col("product_name"),
    col("category"),
    col("price").cast("double")
)

df_stores = df_stores.select(
    col("store_id").cast("int"),
    col("store_name"),
    col("location")
)

df_customers = df_customers.select(
    "customer_id", "first_name", "last_name", "email", "city", "registration_date").dropDuplicates(["customer_id"])

In [0]:
df_silver = df_transactions \
    .join(df_customers, "customer_id") \
        .join(df_products, "product_id") \
            .join(df_stores, "store_id") \
                .withColumn("total_amount", col("quantity") * col("price"))

In [0]:
df_silver.display()

In [0]:
silver_path = "/mnt/retail_project/silver/"
df_silver.write.mode("overwrite").format("delta").save(silver_path)

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS retail_silver_cleaned
          USING DELTA
          LOCATION '/mnt/retail_project/silver/'
          """)

In [0]:
%sql
SELECT * FROM retail_silver_cleaned

In [0]:
silver_df = spark.read.format("delta").load("/mnt/retail_project/silver/")

### **Gold Layer**

In [0]:
from pyspark.sql.functions import sum,countDistinct , avg
gold_df = silver_df.groupBy(
"transaction_date",
"product_id","product_name","category", "store_id","store_name", "location"
).agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("total_amount").alias("total_sales_amount"),
    countDistinct("transaction_id").alias("number_of_transactions"),
    avg("total_amount").alias("average_transaction_value")
)

In [0]:
gold_df.display()

In [0]:
gold_path = "/mnt/retail_project/gold/"
gold_df.write.mode("overwrite").format("delta").save(gold_path)

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS retail_gold_sales_summary
          USING DELTA
          LOCATION '/mnt/retail_project/gold/'
          """)

In [0]:
%sql
SELECT * FROM retail_gold_sales_summary